In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("debarshichanda/goemotions")

print("Path to dataset files:", path)

c:\Users\aashr\OneDrive\Desktop\HackShastra_Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\aashr\.cache\kagglehub\datasets\debarshichanda\goemotions\versions\6


In [2]:
import os

print(os.listdir(path + "/data"))

['dev.tsv', 'ekman_labels.csv', 'ekman_mapping.json', 'emotions.txt', 'full_dataset', 'sentiment_dict.json', 'sentiment_mapping.json', 'test.tsv', 'train.tsv']


In [3]:
import pandas as pd
import numpy as np

In [4]:
path=path + '/data'
df = pd.read_csv(
    path + "/train.tsv",
    sep="\t",
    header=None,
    names=["text", "labels", "id"]
)

print(df.head())

                                                text labels       id
0  My favourite food is anything I didn't have to...     27  eebbqej
1  Now if he does off himself, everyone will thin...     27  ed00q6i
2                     WHY THE FUCK IS BAYLESS ISOING      2  eezlygj
3                        To make her feel threatened     14  ed7ypvh
4                             Dirty Southern Wankers      3  ed0bdzj


In [5]:
with open(path + "/emotions.txt", "r") as f:
    emotions = [line.strip() for line in f.readlines()]

print(emotions[:10])

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment']


In [6]:
emotion_map = {i: emotion for i, emotion in enumerate(emotions)}

In [7]:
df.columns

Index(['text', 'labels', 'id'], dtype='str')

In [8]:
def decode_labels(label_str):
    ids = list(map(int, label_str.split(",")))
    return [emotion_map[i] for i in ids]

df["emotion_names"] = df["labels"].apply(decode_labels)

In [9]:
print(df[["text", "emotion_names"]].head())

                                                text emotion_names
0  My favourite food is anything I didn't have to...     [neutral]
1  Now if he does off himself, everyone will thin...     [neutral]
2                     WHY THE FUCK IS BAYLESS ISOING       [anger]
3                        To make her feel threatened        [fear]
4                             Dirty Southern Wankers   [annoyance]


In [10]:
def map_to_mental_health(emotions):
    stress = 0
    anxiety = 0
    depression = 0

    if any(e in emotions for e in ["sadness", "grief", "disappointment", "hopelessness"]):
        depression = 1

    if any(e in emotions for e in ["fear", "nervousness"]):
        anxiety = 1

    if any(e in emotions for e in ["anger", "annoyance", "frustration"]):
        stress = 1

    return stress, depression, anxiety

In [11]:
df[["stress", "depression", "anxiety"]] = df["emotion_names"].apply(
    lambda x: pd.Series(map_to_mental_health(x))
)

In [12]:
final_df = df[["text", "stress", "depression", "anxiety"]]
final_df

,text,stress,depression,anxiety
0,My favourite food is anything I didn't have to...,0,0,0
1,"Now if he does off himself, everyone will thin...",0,0,0
2,WHY THE FUCK IS BAYLESS ISOING,1,0,0
3,To make her feel threatened,0,0,1
4,Dirty Southern Wankers,1,0,0
...,...,...,...,...
43405,Added you mate well I’ve just got the bow and ...,0,0,0
43406,Always thought that was funny but is it a refe...,0,0,0
43407,What are you talking about? Anything bad that ...,1,0,0
43408,"More like a baptism, with sexy results!",0,0,0


In [13]:
np.sum(final_df['depression']==1)

np.int64(2513)

In [14]:
np.sum(final_df['stress']==1)

np.int64(3768)

In [15]:
np.sum(final_df['anxiety']==1)

np.int64(726)

In [16]:
final_df.to_csv("mental_health_dataset.csv", index=False)

In [75]:
from sklearn.utils import resample

# Oversample to match stress count (~3768)
anxiety_oversampled = resample(
    final_df[final_df['anxiety'] == 1],
    replace=True,          # sample with replacement
    n_samples=3768,        # match largest class
    random_state=42
)

In [76]:
import torch

# Calculate class weights
counts = final_df[["stress", "depression", "anxiety"]].sum()

total = len(final_df)

weights = total / (3 * counts)
print(weights)

stress         3.840234
depression     5.758058
anxiety       19.931129
dtype: float64


In [77]:
from torch.nn import BCEWithLogitsLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_weights = torch.tensor(weights.values, dtype=torch.float).to(device)

def compute_loss(model, inputs, return_outputs=False, **kwargs):
    labels = inputs.pop("labels").to(device)   # move labels to GPU
    
    outputs = model(**inputs)
    logits = outputs.logits

    loss_fn = BCEWithLogitsLoss(pos_weight=class_weights)
    loss = loss_fn(logits, labels.float())

    return (loss, outputs) if return_outputs else loss

In [78]:
from datasets import Dataset

dataset = Dataset.from_pandas(final_df)

# Split data
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [79]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map: 100%|██████████| 8682/8682 [00:01<00:00, 5252.40 examples/s]


In [80]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "stress", "depression", "anxiety"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "stress", "depression", "anxiety"]
)

In [81]:
import torch

def format_labels(example):
    example["labels"] = torch.tensor([
        example["stress"],
        example["depression"],
        example["anxiety"]
    ])
    return example

train_dataset = train_dataset.map(format_labels)
test_dataset = test_dataset.map(format_labels)

Map: 100%|██████████| 8682/8682 [00:12<00:00, 677.24 examples/s]


In [82]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    problem_type="multi_label_classification"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4348.82it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [83]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [84]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    return {
        "f1": f1_score(labels, preds, average="micro")
    }

In [86]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# Inject your custom loss
trainer.compute_loss = compute_loss

In [87]:
import torch
print(torch.cuda.is_available())

True


In [88]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [89]:
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [90]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.504255,0.644261,0.442843
2,0.424823,0.523223,0.543613
3,0.272064,0.674409,0.538959


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.43it/s]


TrainOutput(global_step=13023, training_loss=0.41920534388898, metrics={'train_runtime': 1215.0242, 'train_samples_per_second': 85.746, 'train_steps_per_second': 10.718, 'total_flos': 3450307395631104.0, 'train_loss': 0.41920534388898, 'epoch': 3.0})

In [91]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = inputs.to(device)

    outputs = model(**inputs)
    logits = outputs.logits.detach().cpu().numpy()[0]

    probs = 1 / (1 + np.exp(-logits))

    return {
        "stress": probs[0],
        "depression": probs[1],
        "anxiety": probs[2]
    }

In [92]:
THRESHOLDS = {
    "stress": 0.4,
    "depression": 0.4,
    "anxiety": 0.3   # lower because rare class
}

In [93]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

probs = 1 / (1 + np.exp(-logits))
preds = (probs > 0.5).astype(int)

print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.52      0.55      0.54       749
           1       0.52      0.52      0.52       528
           2       0.55      0.67      0.61       153

   micro avg       0.52      0.55      0.54      1430
   macro avg       0.53      0.58      0.55      1430
weighted avg       0.52      0.55      0.54      1430
 samples avg       0.09      0.09      0.09      1430



c:\Users\aashr\OneDrive\Desktop\HackShastra_Hackathon\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\aashr\OneDrive\Desktop\HackShastra_Hackathon\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\aashr\OneDrive\Desktop\HackShastra_Hackathon\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
 

In [94]:
def generate_report(text):
    result = predict(text)

    report = {}

    report["Stress Risk"] = "High" if result["stress"] > 0.7 else "Moderate" if result["stress"] > 0.4 else "Low"
    report["Depression Risk"] = "High" if result["depression"] > 0.7 else "Moderate" if result["depression"] > 0.4 else "Low"
    report["Anxiety Risk"] = "High" if result["anxiety"] > 0.5 else "Moderate" if result["anxiety"] > 0.3 else "Low"

    if max(result.values()) > 0.7:
        report["Action"] = "Mental health consultation recommended"
    else:
        report["Action"] = "Monitor and maintain healthy lifestyle"

    return report

In [95]:
print(generate_report("I feel exhausted and overwhelmed lately"))

{'Stress Risk': 'Low', 'Depression Risk': 'High', 'Anxiety Risk': 'Low', 'Action': 'Mental health consultation recommended'}
